In [ ]:
#| default_exp inference

In [ ]:
#| export
import torch
import torch.nn.functional as F
from huggingface_hub import hf_hub_download

from contextlib import nullcontext

In [ ]:
#| exporti
def get_default_compute_device():
    if torch.cuda.is_available() and (torch.version.cuda or torch.version.hip):
        return 'cuda'
    elif torch.backends.mps.is_available():
        return 'mps'
    else:
        return 'cpu'

preferred_device = None

In [ ]:
#| export
def get_compute_device():
    global preferred_device
    if preferred_device is None: preferred_device = get_default_compute_device()
    return preferred_device

In [ ]:
#| exporti
def load_model(ref=None, spec=None, device='cpu'):
    if spec is not None: return spec
    if ":" in ref:
        repo_id, filename = ref.split(":", 1)
        local_filename = hf_hub_download(repo_id=repo_id, filename=filename)
    else:
        local_filename = ref
    return torch.load(local_filename, map_location=device)

In [ ]:
#| exporti

def inference_context():
    if torch.cuda.is_available():
        return torch.backends.cuda.sdp_kernel(enable_flash=False, enable_mem_efficient=False, enable_math=True)
    else:
        return nullcontext()

# from https://github.com/pytorch-labs/gpt-fast/blob/main/generate.py
def multinomial_sample_one_no_sync(probs_sort): # Does multinomial sampling without a cuda synchronization
    q = torch.empty_like(probs_sort).exponential_(1)
    return torch.argmax(probs_sort / q, dim=-1, keepdim=True).to(dtype=torch.int)

def logits_to_probs(logits, T=1.0, top_k=None):
    logits = logits / max(T, 1e-5)

    if top_k is not None:
        v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
        pivot = v.select(-1, -1).unsqueeze(-1)
        logits = torch.where(logits < pivot, -float("Inf"), logits)

    probs = torch.nn.functional.softmax(logits, dim=-1)
    return probs

def sample(logits, T=1.0, top_k=None):
    probs = logits_to_probs(logits, T, top_k)
    idx_next = multinomial_sample_one_no_sync(probs)
    return idx_next

In [ ]:
#| export
def load_audio(fname, max_seconds=None):
    "Load an audio file as a `(channels, samples)` float32 tensor and its sample rate (like `torchaudio.load`)."
    # soundfile instead of torchaudio.load/info: torchaudio 2.9 removed `info` and routes `load` through
    # torchcodec, which needs matching FFmpeg shared libraries and fails on a plain install.
    import soundfile as sf
    with sf.SoundFile(str(fname)) as f:
        frames = -1 if max_seconds is None else int(max_seconds * f.samplerate)
        data = f.read(frames=frames, dtype='float32', always_2d=True)
        return torch.from_numpy(data.T.copy()), f.samplerate

In [ ]:
# Regression test: load_audio reads (channels, samples) float32 like torchaudio.load, without FFmpeg
import os, tempfile, numpy as np, soundfile as sf
_fails = []
with tempfile.TemporaryDirectory() as d:
    _st = np.stack([np.full(48000, .25), np.full(48000, -.5)], axis=1).astype('float32')  # 2 s stereo, 24 kHz
    for _ext in ['wav', 'flac', 'ogg', 'mp3']:
        fn = os.path.join(d, f's.{_ext}'); sf.write(fn, _st, 24000)
        try:
            _x, _sr = load_audio(fn)
            if _sr != 24000 or _x.shape[0] != 2 or _x.dtype != torch.float32: _fails.append(f'{_ext}: {_sr} {tuple(_x.shape)} {_x.dtype}')
        except Exception as e: _fails.append(f'{_ext}: {type(e).__name__}: {str(e).splitlines()[0][:80]}')
    try:
        _x, _ = load_audio(os.path.join(d, 's.wav'), max_seconds=0.5)
        if tuple(_x.shape) != (2, 12000) or abs(float(_x[1, 0]) + .5) > 1e-4: _fails.append(f'max_seconds/values: {tuple(_x.shape)}')
    except Exception as e: _fails.append(f'max_seconds: {type(e).__name__}: {e}')
assert not _fails, 'load_audio: ' + '; '.join(_fails)